# Notebook 08 — Macroeconomic Data Preparation (Version 2)

Prepare four quarterly macroeconomic predictors and map each reference
quarter-end $t$ to source quarter $t-1$ exactly once. The full macro panel is
derived from `REFERENCE_CALENDAR_START` and the common observed source-data
end; no quarter count is hard-coded.

The Notebook 09 candidate calendar is derived independently from Notebook 07
after excluding dates before `REFERENCE_CALENDAR_START`. Current-vintage FRED
snapshots are used, so the values may incorporate revisions made after the
historical reference dates. Publication timing is therefore a nominal
availability check, not a real-time-vintage reconstruction.

In [ ]:
# Lock paths, transformations and the one-lag timing contract

from pathlib import Path
import hashlib
import importlib.util
import os

import numpy as np
import pandas as pd
from IPython.display import display


try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass


PROJECT_ROOT = Path(
    os.environ.get(
        "DISSERTATION_PROJECT_ROOT",
        os.environ.get("DATA_FINAL_ROOT", "/content/drive/MyDrive/Data Final"),
    )
).expanduser().resolve()

explicit_config = os.environ.get("V2_CONFIG_PATH")
config_candidates = [
    Path(explicit_config).expanduser() if explicit_config else None,
    PROJECT_ROOT / "config_v2.py",
    PROJECT_ROOT / "config.py",
    Path.cwd() / "config_v2.py",
    Path.cwd() / "config.py",
]
config_matches = []
for candidate in config_candidates:
    if candidate is not None and candidate.is_file():
        resolved = candidate.resolve()
        if resolved not in config_matches:
            config_matches.append(resolved)
if not config_matches:
    raise FileNotFoundError(
        "No shared configuration was found. Expected config_v2.py or config.py "
        f"under {PROJECT_ROOT}, or set V2_CONFIG_PATH explicitly."
    )
CONFIG_PATH = config_matches[0]

config_spec = importlib.util.spec_from_file_location(
    "dissertation_config_v2",
    CONFIG_PATH,
)
config_v2 = importlib.util.module_from_spec(config_spec)
config_spec.loader.exec_module(config_v2)

ANALYSIS_VERSION = getattr(config_v2, "ANALYSIS_VERSION", "v2_bank_lag1_main")
REFERENCE_CALENDAR_START = pd.Timestamp(
    getattr(config_v2, "REFERENCE_CALENDAR_START", "2001-06-30")
)
TARGET_HORIZON_DAYS = int(getattr(config_v2, "TARGET_HORIZON_DAYS", 365))
MACRO_LAG_APPLICATIONS = 1
PREDICTOR_TIMING_POLICY = "B(t-1), M(t-1) -> Y(t, t+365 days]"
MACRO_TIMING_POLICY = (
    "Macro predictors are mapped from source quarter t-1 to reference quarter t "
    "exactly once; do not lag them again when bank predictors are shifted to t-1."
)

MACRO_RAW_DIR = PROJECT_ROOT / "Raw Data" / "Macro"
NB07_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Notebook Outputs"
    / ANALYSIS_VERSION
    / "07_One_Year_Failure_Target"
)
NB08_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Notebook Outputs"
    / ANALYSIS_VERSION
    / "08_Macroeconomic_Data_Preparation"
)
NB08_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_HANDOFF_PATH = (
    NB07_OUTPUT_DIR
    / "bank_failure_target_1yr_2001Q1_2025Q2.parquet"
)
MACRO_SOURCE_PATHS = {
    "GDPC1": MACRO_RAW_DIR / "real_gdp_raw.csv",
    "UNRATE": MACRO_RAW_DIR / "unemployment_rate_raw.csv",
    "DFF": MACRO_RAW_DIR / "effective_federal_funds_rate_raw.csv",
    "PONHPIM226S": MACRO_RAW_DIR / "house_price_index_raw.csv",
}

required_paths = [CONFIG_PATH, TARGET_HANDOFF_PATH] + list(MACRO_SOURCE_PATHS.values())
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Notebook 08 prerequisites were not found:\n" + "\n".join(missing_paths)
    )

MACRO_PREDICTORS = [
    "real_gdp_growth_lag1",
    "unemployment_rate_lag1",
    "effective_federal_funds_rate_lag1",
    "house_price_growth_lag1",
]

macro_definition = pd.DataFrame({
    "series_id": ["GDPC1", "UNRATE", "DFF", "PONHPIM226S"],
    "source_measure": [
        "Real gross domestic product",
        "Civilian unemployment rate",
        "Effective federal funds rate",
        "Purchase Only House Price Index for the United States",
    ],
    "raw_frequency": ["Quarterly", "Monthly", "Daily", "Quarterly"],
    "quarterly_transformation": [
        "100 × (current-quarter level / previous-quarter level − 1); QoQ and not annualised",
        "Arithmetic mean of the three monthly observations",
        "Arithmetic mean of all available daily observations",
        "100 × (current-quarter index / previous-quarter index − 1); QoQ and not annualised",
    ],
    "final_predictor": MACRO_PREDICTORS,
    "timing_rule": ["Source quarter t−1 mapped once to reference quarter t"] * 4,
    "nominal_publication_lag_days": [35, 10, 3, 60],
    "nominal_publication_basis": [
        "BEA advance GDP estimate is normally released about one month after quarter-end",
        "The final month of the quarterly mean is normally released early in the following month",
        "The prior business day's EFFR is normally published the next business morning",
        "Quarterly FHFA HPI is normally released about two months after quarter-end",
    ],
    "official_timing_source": [
        "https://www.bea.gov/resources/learning-center/what-to-know-gdp",
        "https://www.bls.gov/cps/definitions.htm",
        "https://www.newyorkfed.org/markets/reference-rates/effr",
        "https://www.fhfa.gov/data/hpi",
    ],
    "source_url": [
        "https://fred.stlouisfed.org/series/GDPC1",
        "https://fred.stlouisfed.org/series/UNRATE",
        "https://fred.stlouisfed.org/series/DFF",
        "https://fred.stlouisfed.org/series/PONHPIM226S",
    ],
})

cell_01_checks = {
    "The analysis version is the V2 lagged-bank main design":
        (ANALYSIS_VERSION == "v2_bank_lag1_main"),
    "The Notebook 07 V2 target handoff exists": TARGET_HANDOFF_PATH.exists(),
    "All four audited macro raw files exist": all(
        path.exists() for path in MACRO_SOURCE_PATHS.values()
    ),
    "Exactly four macro predictors are defined": len(MACRO_PREDICTORS) == 4,
    "The V2 reference calendar begins in 2001Q2":
        (REFERENCE_CALENDAR_START == pd.Timestamp("2001-06-30")),
    "The target horizon remains 365 days": (TARGET_HORIZON_DAYS == 365),
    "Every macro predictor is explicitly marked lag1": all(
        name.endswith("_lag1") for name in MACRO_PREDICTORS
    ),
    "The macro lag is applied exactly once": (MACRO_LAG_APPLICATIONS == 1),
    "GDP and house prices use non-annualised quarter-on-quarter growth": bool(
        macro_definition.loc[
            macro_definition["series_id"].isin(["GDPC1", "PONHPIM226S"]),
            "quarterly_transformation",
        ].str.contains("not annualised", case=False, regex=False).all()
    ),
    "Nominal publication timing is registered for every series": bool(
        macro_definition["nominal_publication_lag_days"].notna().all()
    ),
}

notebook_08_cell_01_gate = pd.DataFrame({
    "check": list(cell_01_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_01_checks.values()],
    "passed": list(cell_01_checks.values()),
})
macro_definition.to_csv(
    NB08_OUTPUT_DIR / "cell_01_macro_definition_register.csv",
    index=False,
)
notebook_08_cell_01_gate.to_csv(
    NB08_OUTPUT_DIR / "cell_01_setup_gate.csv",
    index=False,
)

print("Notebook 08 Cell 1 validation gate:")
display(notebook_08_cell_01_gate)
print("\nLocked macroeconomic definitions and nominal publication assumptions:")
with pd.option_context("display.max_colwidth", None):
    display(macro_definition)

failed = [name for name, passed in cell_01_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 08 Cell 1 blocked. Failed checks: " + "; ".join(failed))

print("\nCell 1 completed successfully.")
print("Shared configuration:", CONFIG_PATH)
print("Analysis version:", ANALYSIS_VERSION)
print("Predictor timing:", PREDICTOR_TIMING_POLICY)
print("Notebook 07 calendar source:", TARGET_HANDOFF_PATH)
print("Output directory:", NB08_OUTPUT_DIR)

In [ ]:
# Load raw FRED sources and derive the V2 candidate reference calendar

if not notebook_08_cell_01_gate["passed"].all():
    raise AssertionError("Cell 2 blocked because Cell 1 did not pass.")


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def load_fred_series(path, expected_series_id):
    raw = pd.read_csv(path)
    if raw.empty:
        raise ValueError(f"{path.name} is empty.")

    lower_lookup = {str(column).strip().lower(): column for column in raw.columns}
    date_column = next(
        (lower_lookup[name] for name in ["observation_date", "date"] if name in lower_lookup),
        None,
    )
    if date_column is None:
        raise KeyError(f"{path.name} has no DATE or observation_date column.")

    exact_series_column = next(
        (
            column for column in raw.columns
            if str(column).strip().upper() == expected_series_id.upper()
        ),
        None,
    )
    if exact_series_column is not None:
        value_column = exact_series_column
    elif "value" in lower_lookup:
        value_column = lower_lookup["value"]
    else:
        remaining = [column for column in raw.columns if column != date_column]
        if len(remaining) != 1:
            raise KeyError(
                f"Could not identify {expected_series_id} in {path.name}. "
                f"Columns found: {list(raw.columns)}"
            )
        value_column = remaining[0]

    date_text = raw[date_column].astype("string").str.strip()
    parsed_date = pd.to_datetime(date_text, errors="coerce")
    numeric_value = pd.to_numeric(
        raw[value_column].replace({".": np.nan, "": np.nan}),
        errors="coerce",
    )
    cleaned = pd.DataFrame({"date": parsed_date, "value": numeric_value})
    metadata = {
        "series_id": expected_series_id,
        "filename": path.name,
        "sha256": sha256_file(path),
        "raw_rows": len(raw),
        "raw_columns": len(raw.columns),
        "date_column": str(date_column),
        "value_column": str(value_column),
        "invalid_date_rows": int((date_text.notna() & parsed_date.isna()).sum()),
        "duplicate_dates": int(cleaned["date"].dropna().duplicated().sum()),
        "missing_numeric_values": int(numeric_value.isna().sum()),
        "first_parsed_date": parsed_date.min(),
        "last_parsed_date": parsed_date.max(),
        "last_observed_quarter": parsed_date.max().to_period("Q").end_time.normalize(),
    }
    return cleaned.sort_values("date").reset_index(drop=True), metadata


macro_raw = {}
source_records = []
for series_id, path in MACRO_SOURCE_PATHS.items():
    macro_raw[series_id], record = load_fred_series(path, series_id)
    source_records.append(record)
cell_02_source_manifest = pd.DataFrame(source_records)

notebook_07_reference_series = pd.read_parquet(
    TARGET_HANDOFF_PATH,
    columns=["report_date"],
)["report_date"]
notebook_07_reference_series = pd.to_datetime(
    notebook_07_reference_series,
    errors="coerce",
)
NOTEBOOK_07_REFERENCE_CALENDAR = pd.DatetimeIndex(
    sorted(notebook_07_reference_series.dropna().unique())
)
CANDIDATE_REFERENCE_CALENDAR = NOTEBOOK_07_REFERENCE_CALENDAR[
    NOTEBOOK_07_REFERENCE_CALENDAR >= REFERENCE_CALENDAR_START
]
EXCLUDED_REFERENCE_CALENDAR = NOTEBOOK_07_REFERENCE_CALENDAR[
    NOTEBOOK_07_REFERENCE_CALENDAR < REFERENCE_CALENDAR_START
]

expected_notebook_07_calendar = pd.date_range(
    NOTEBOOK_07_REFERENCE_CALENDAR.min(),
    NOTEBOOK_07_REFERENCE_CALENDAR.max(),
    freq="QE-DEC",
)
expected_candidate_calendar = pd.date_range(
    CANDIDATE_REFERENCE_CALENDAR.min(),
    CANDIDATE_REFERENCE_CALENDAR.max(),
    freq="QE-DEC",
)

cell_02_reference_calendar_audit = pd.DataFrame({
    "measure": [
        "notebook_07_reference_dates",
        "reference_calendar_start",
        "restricted_candidate_reference_dates",
        "notebook_07_reference_dates_excluded",
        "excluded_reference_dates",
        "first_candidate_reference_date",
        "last_candidate_reference_date",
    ],
    "value": [
        len(NOTEBOOK_07_REFERENCE_CALENDAR),
        REFERENCE_CALENDAR_START,
        len(CANDIDATE_REFERENCE_CALENDAR),
        len(EXCLUDED_REFERENCE_CALENDAR),
        ",".join(str(date.to_period("Q")) for date in EXCLUDED_REFERENCE_CALENDAR),
        CANDIDATE_REFERENCE_CALENDAR.min(),
        CANDIDATE_REFERENCE_CALENDAR.max(),
    ],
})

cell_02_checks = {
    "Exactly four raw macro series were loaded": len(macro_raw) == 4,
    "All expected FRED identifiers were loaded": set(macro_raw) == set(MACRO_SOURCE_PATHS),
    "No raw source has an invalid date":
        (int(cell_02_source_manifest["invalid_date_rows"].sum()) == 0),
    "No raw source contains a duplicate observation date":
        (int(cell_02_source_manifest["duplicate_dates"].sum()) == 0),
    "Every raw source contains numeric observations": all(
        frame["value"].notna().any() for frame in macro_raw.values()
    ),
    "Notebook 07 reference dates are continuous quarter-ends":
        NOTEBOOK_07_REFERENCE_CALENDAR.equals(expected_notebook_07_calendar),
    "The restricted candidate calendar is continuous":
        CANDIDATE_REFERENCE_CALENDAR.equals(expected_candidate_calendar),
    "The candidate calendar begins at REFERENCE_CALENDAR_START":
        (CANDIDATE_REFERENCE_CALENDAR.min() == REFERENCE_CALENDAR_START),
    "Exactly one Notebook 07 reference date is excluded":
        (len(EXCLUDED_REFERENCE_CALENDAR) == 1),
    "The excluded reference date is exactly 2001Q1": bool(
        len(EXCLUDED_REFERENCE_CALENDAR) == 1
        and EXCLUDED_REFERENCE_CALENDAR[0] == pd.Timestamp("2001-03-31")
    ),
    "Notebook 07 contains no missing reference date":
        (int(notebook_07_reference_series.isna().sum()) == 0),
}

notebook_08_cell_02_gate = pd.DataFrame({
    "check": list(cell_02_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_02_checks.values()],
    "passed": list(cell_02_checks.values()),
})
cell_02_source_manifest.to_csv(
    NB08_OUTPUT_DIR / "cell_02_raw_macro_source_manifest.csv",
    index=False,
)
pd.DataFrame({"report_date": NOTEBOOK_07_REFERENCE_CALENDAR}).to_csv(
    NB08_OUTPUT_DIR / "cell_02_notebook_07_reference_calendar.csv",
    index=False,
)
pd.DataFrame({"report_date": CANDIDATE_REFERENCE_CALENDAR}).to_csv(
    NB08_OUTPUT_DIR / "cell_02_v2_candidate_reference_calendar.csv",
    index=False,
)
cell_02_reference_calendar_audit.to_csv(
    NB08_OUTPUT_DIR / "cell_02_reference_calendar_restriction_audit.csv",
    index=False,
)
notebook_08_cell_02_gate.to_csv(
    NB08_OUTPUT_DIR / "cell_02_input_validation_gate.csv",
    index=False,
)

print("Notebook 08 Cell 2 validation gate:")
display(notebook_08_cell_02_gate)
print("\nRaw macro source manifest:")
display(cell_02_source_manifest)
print("\nReference-calendar restriction audit:")
display(cell_02_reference_calendar_audit)

failed = [name for name, passed in cell_02_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 08 Cell 2 blocked. Failed checks: " + "; ".join(failed))

print("\nCell 2 completed successfully.")

In [ ]:
# Convert sources to quarterly frequency and derive the full calendar

if not notebook_08_cell_02_gate["passed"].all():
    raise AssertionError("Cell 3 blocked because Cell 2 did not pass.")


def add_quarter_end(frame):
    result = frame.dropna(subset=["date"]).copy()
    result["quarter_end"] = result["date"].dt.to_period("Q").dt.to_timestamp("Q")
    return result


gdp_raw_q = add_quarter_end(macro_raw["GDPC1"])
unemployment_raw_q = add_quarter_end(macro_raw["UNRATE"])
fed_funds_raw_q = add_quarter_end(macro_raw["DFF"])
house_price_raw_q = add_quarter_end(macro_raw["PONHPIM226S"])

gdp_quarterly = gdp_raw_q.groupby("quarter_end")["value"].agg(["last", "count"])
unemployment_quarterly = unemployment_raw_q.groupby("quarter_end")["value"].agg(["mean", "count"])
fed_funds_quarterly = fed_funds_raw_q.groupby("quarter_end")["value"].agg(["mean", "count"])
house_price_quarterly = house_price_raw_q.groupby("quarter_end")["value"].agg(["last", "count"])

latest_complete_source_quarter = {
    "GDPC1": gdp_quarterly.index[gdp_quarterly["count"].eq(1)].max(),
    "UNRATE": unemployment_quarterly.index[unemployment_quarterly["count"].eq(3)].max(),
    "DFF": fed_funds_quarterly.index[
        fed_funds_quarterly["count"].gt(0)
        & fed_funds_quarterly.index.to_series().le(
            macro_raw["DFF"]["date"].max().normalize()
        ).to_numpy()
    ].max(),
    "PONHPIM226S": house_price_quarterly.index[
        house_price_quarterly["count"].eq(1)
    ].max(),
}
SOURCE_CALENDAR_START = REFERENCE_CALENDAR_START - pd.offsets.QuarterEnd(1)
LATEST_COMMON_COMPLETE_CANDIDATE = min(latest_complete_source_quarter.values())
completeness_probe_calendar = pd.date_range(
    SOURCE_CALENDAR_START,
    LATEST_COMMON_COMPLETE_CANDIDATE,
    freq="QE-DEC",
)
source_quarter_completeness = pd.DataFrame(index=completeness_probe_calendar)
source_quarter_completeness.index.name = "source_quarter"
source_quarter_completeness["GDPC1_complete"] = (
    gdp_quarterly["count"].reindex(completeness_probe_calendar).eq(1)
)
source_quarter_completeness["UNRATE_complete"] = (
    unemployment_quarterly["count"].reindex(completeness_probe_calendar).eq(3)
)
source_quarter_completeness["DFF_complete"] = (
    fed_funds_quarterly["count"].reindex(completeness_probe_calendar).gt(0)
    & completeness_probe_calendar.to_series().le(
        macro_raw["DFF"]["date"].max().normalize()
    ).to_numpy()
)
source_quarter_completeness["PONHPIM226S_complete"] = (
    house_price_quarterly["count"].reindex(completeness_probe_calendar).eq(1)
)
completeness_columns = [
    "GDPC1_complete",
    "UNRATE_complete",
    "DFF_complete",
    "PONHPIM226S_complete",
]
source_quarter_completeness["all_four_series_complete"] = (
    source_quarter_completeness[completeness_columns].all(axis=1)
)
incomplete_source_quarters = source_quarter_completeness.loc[
    ~source_quarter_completeness["all_four_series_complete"]
].reset_index()
if incomplete_source_quarters.empty:
    FIRST_INCOMPLETE_SOURCE_QUARTER = pd.NaT
    COMMON_SOURCE_DATA_END = LATEST_COMMON_COMPLETE_CANDIDATE
else:
    FIRST_INCOMPLETE_SOURCE_QUARTER = incomplete_source_quarters[
        "source_quarter"
    ].min()
    COMMON_SOURCE_DATA_END = (
        FIRST_INCOMPLETE_SOURCE_QUARTER - pd.offsets.QuarterEnd(1)
    )

FULL_REPORT_END = COMMON_SOURCE_DATA_END + pd.offsets.QuarterEnd(1)
FULL_REPORT_CALENDAR = pd.date_range(
    REFERENCE_CALENDAR_START,
    FULL_REPORT_END,
    freq="QE-DEC",
)
FULL_SOURCE_CALENDAR = pd.DatetimeIndex([
    date - pd.offsets.QuarterEnd(1) for date in FULL_REPORT_CALENDAR
])
LEVEL_COMPONENT_START = FULL_SOURCE_CALENDAR.min() - pd.offsets.QuarterEnd(1)
LEVEL_COMPONENT_CALENDAR = pd.date_range(
    LEVEL_COMPONENT_START,
    COMMON_SOURCE_DATA_END,
    freq="QE-DEC",
)

quarterly_components = pd.DataFrame(index=LEVEL_COMPONENT_CALENDAR)
quarterly_components.index.name = "quarter_end"
quarterly_components["real_gdp_level"] = gdp_quarterly["last"].reindex(LEVEL_COMPONENT_CALENDAR)
quarterly_components["unemployment_rate"] = unemployment_quarterly["mean"].reindex(LEVEL_COMPONENT_CALENDAR)
quarterly_components["effective_federal_funds_rate"] = fed_funds_quarterly["mean"].reindex(LEVEL_COMPONENT_CALENDAR)
quarterly_components["house_price_index"] = house_price_quarterly["last"].reindex(LEVEL_COMPONENT_CALENDAR)
quarterly_components["gdp_observations"] = gdp_quarterly["count"].reindex(LEVEL_COMPONENT_CALENDAR).fillna(0).astype(int)
quarterly_components["unemployment_observations"] = unemployment_quarterly["count"].reindex(LEVEL_COMPONENT_CALENDAR).fillna(0).astype(int)
quarterly_components["fed_funds_observations"] = fed_funds_quarterly["count"].reindex(LEVEL_COMPONENT_CALENDAR).fillna(0).astype(int)
quarterly_components["house_price_observations"] = house_price_quarterly["count"].reindex(LEVEL_COMPONENT_CALENDAR).fillna(0).astype(int)

cell_03_observed_end_audit = pd.DataFrame({
    "series_id": list(latest_complete_source_quarter),
    "latest_complete_source_quarter": list(latest_complete_source_quarter.values()),
})
cell_03_observed_end_audit["latest_common_complete_candidate"] = (
    LATEST_COMMON_COMPLETE_CANDIDATE
)
cell_03_observed_end_audit["first_incomplete_source_quarter"] = (
    FIRST_INCOMPLETE_SOURCE_QUARTER
)
cell_03_observed_end_audit["common_source_data_end"] = COMMON_SOURCE_DATA_END
cell_03_observed_end_audit["derived_full_report_end"] = FULL_REPORT_END
cell_03_observed_end_audit["truncated_at_first_internal_gap"] = bool(
    pd.notna(FIRST_INCOMPLETE_SOURCE_QUARTER)
)

cell_03_frequency_audit = pd.DataFrame({
    "series_id": ["GDPC1", "UNRATE", "DFF", "PONHPIM226S"],
    "aggregation": [
        "Quarterly level",
        "Quarterly arithmetic mean",
        "Quarterly arithmetic mean",
        "Quarterly level",
    ],
    "required_first_quarter": [
        LEVEL_COMPONENT_CALENDAR.min(),
        FULL_SOURCE_CALENDAR.min(),
        FULL_SOURCE_CALENDAR.min(),
        LEVEL_COMPONENT_CALENDAR.min(),
    ],
    "required_last_quarter": [COMMON_SOURCE_DATA_END] * 4,
    "missing_required_quarters": [
        int(quarterly_components.loc[LEVEL_COMPONENT_CALENDAR, "real_gdp_level"].isna().sum()),
        int(quarterly_components.loc[FULL_SOURCE_CALENDAR, "unemployment_rate"].isna().sum()),
        int(quarterly_components.loc[FULL_SOURCE_CALENDAR, "effective_federal_funds_rate"].isna().sum()),
        int(quarterly_components.loc[LEVEL_COMPONENT_CALENDAR, "house_price_index"].isna().sum()),
    ],
    "minimum_observations_per_required_quarter": [
        int(quarterly_components.loc[LEVEL_COMPONENT_CALENDAR, "gdp_observations"].min()),
        int(quarterly_components.loc[FULL_SOURCE_CALENDAR, "unemployment_observations"].min()),
        int(quarterly_components.loc[FULL_SOURCE_CALENDAR, "fed_funds_observations"].min()),
        int(quarterly_components.loc[LEVEL_COMPONENT_CALENDAR, "house_price_observations"].min()),
    ],
    "maximum_observations_per_required_quarter": [
        int(quarterly_components.loc[LEVEL_COMPONENT_CALENDAR, "gdp_observations"].max()),
        int(quarterly_components.loc[FULL_SOURCE_CALENDAR, "unemployment_observations"].max()),
        int(quarterly_components.loc[FULL_SOURCE_CALENDAR, "fed_funds_observations"].max()),
        int(quarterly_components.loc[LEVEL_COMPONENT_CALENDAR, "house_price_observations"].max()),
    ],
})

cell_03_checks = {
    "A common complete source-data end was derived": pd.notna(COMMON_SOURCE_DATA_END),
    "The full report calendar begins at REFERENCE_CALENDAR_START":
        (FULL_REPORT_CALENDAR.min() == REFERENCE_CALENDAR_START),
    "The full report calendar ends one quarter after the common source-data end":
        (COMMON_SOURCE_DATA_END.to_period("Q") + 1 == FULL_REPORT_END.to_period("Q")),
    "The full report calendar is derived without a fixed quarter count":
        FULL_REPORT_CALENDAR.equals(
            pd.date_range(REFERENCE_CALENDAR_START, FULL_REPORT_END, freq="QE-DEC")
        ),
    "The common source calendar stops before the first internal incomplete quarter": bool(
        pd.isna(FIRST_INCOMPLETE_SOURCE_QUARTER)
        or COMMON_SOURCE_DATA_END
        == FIRST_INCOMPLETE_SOURCE_QUARTER - pd.offsets.QuarterEnd(1)
    ),
    "The continuous common source range covers every Notebook 09 candidate date":
        (
            COMMON_SOURCE_DATA_END
            >= CANDIDATE_REFERENCE_CALENDAR.max() - pd.offsets.QuarterEnd(1)
        ),
    "GDP has one observation in every required level quarter": bool(
        quarterly_components.loc[LEVEL_COMPONENT_CALENDAR, "gdp_observations"].eq(1).all()
    ),
    "House prices have one observation in every required level quarter": bool(
        quarterly_components.loc[LEVEL_COMPONENT_CALENDAR, "house_price_observations"].eq(1).all()
    ),
    "Unemployment has three monthly observations in every required source quarter": bool(
        quarterly_components.loc[FULL_SOURCE_CALENDAR, "unemployment_observations"].eq(3).all()
    ),
    "Federal funds data exist in every required source quarter": bool(
        quarterly_components.loc[FULL_SOURCE_CALENDAR, "fed_funds_observations"].gt(0).all()
    ),
    "No required quarterly component is missing":
        (int(cell_03_frequency_audit["missing_required_quarters"].sum()) == 0),
    "The component calendar is continuous":
        pd.DatetimeIndex(quarterly_components.index).equals(LEVEL_COMPONENT_CALENDAR),
}

notebook_08_cell_03_gate = pd.DataFrame({
    "check": list(cell_03_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_03_checks.values()],
    "passed": list(cell_03_checks.values()),
})
quarterly_components.reset_index().to_csv(
    NB08_OUTPUT_DIR / "cell_03_quarterly_components_full_observed_range.csv",
    index=False,
)
cell_03_observed_end_audit.to_csv(
    NB08_OUTPUT_DIR / "cell_03_observed_data_end_audit.csv",
    index=False,
)
incomplete_source_quarters.to_csv(
    NB08_OUTPUT_DIR / "cell_03_incomplete_source_quarter_audit.csv",
    index=False,
)
cell_03_frequency_audit.to_csv(
    NB08_OUTPUT_DIR / "cell_03_frequency_conversion_audit.csv",
    index=False,
)
notebook_08_cell_03_gate.to_csv(
    NB08_OUTPUT_DIR / "cell_03_validation_gate.csv",
    index=False,
)

print("Notebook 08 Cell 3 validation gate:")
display(notebook_08_cell_03_gate)
print("\nObserved-data-end derivation:")
display(cell_03_observed_end_audit)
print("\nFrequency-conversion audit:")
display(cell_03_frequency_audit)

failed = [name for name, passed in cell_03_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 08 Cell 3 blocked. Failed checks: " + "; ".join(failed))

print("\nCell 3 completed successfully.")

In [ ]:
# Calculate growth rates and build the full unlagged source panel

if not notebook_08_cell_03_gate["passed"].all():
    raise AssertionError("Cell 4 blocked because Cell 3 did not pass.")

macro_unlagged_all = pd.DataFrame(index=quarterly_components.index)
macro_unlagged_all.index.name = "source_quarter"
macro_unlagged_all["real_gdp_growth"] = (
    quarterly_components["real_gdp_level"].pct_change(fill_method=None) * 100
)
macro_unlagged_all["unemployment_rate"] = quarterly_components["unemployment_rate"]
macro_unlagged_all["effective_federal_funds_rate"] = quarterly_components[
    "effective_federal_funds_rate"
]
macro_unlagged_all["house_price_growth"] = (
    quarterly_components["house_price_index"].pct_change(fill_method=None) * 100
)
macro_unlagged = macro_unlagged_all.reindex(FULL_SOURCE_CALENDAR).copy()
macro_unlagged.index.name = "source_quarter"

expected_gdp_growth = (
    quarterly_components["real_gdp_level"]
    / quarterly_components["real_gdp_level"].shift(1)
    - 1
) * 100
expected_house_price_growth = (
    quarterly_components["house_price_index"]
    / quarterly_components["house_price_index"].shift(1)
    - 1
) * 100

cell_04_checks = {
    "The unlagged source panel exactly matches the derived full source calendar":
        pd.DatetimeIndex(macro_unlagged.index).equals(FULL_SOURCE_CALENDAR),
    "Real GDP growth exactly matches the locked QoQ formula": bool(
        np.allclose(
            macro_unlagged["real_gdp_growth"].to_numpy(float),
            expected_gdp_growth.reindex(FULL_SOURCE_CALENDAR).to_numpy(float),
            rtol=0,
            atol=1e-12,
            equal_nan=False,
        )
    ),
    "House-price growth exactly matches the locked QoQ formula": bool(
        np.allclose(
            macro_unlagged["house_price_growth"].to_numpy(float),
            expected_house_price_growth.reindex(FULL_SOURCE_CALENDAR).to_numpy(float),
            rtol=0,
            atol=1e-12,
            equal_nan=False,
        )
    ),
    "Unemployment remains the quarterly arithmetic mean": bool(
        np.allclose(
            macro_unlagged["unemployment_rate"].to_numpy(float),
            quarterly_components.loc[FULL_SOURCE_CALENDAR, "unemployment_rate"].to_numpy(float),
            rtol=0,
            atol=1e-12,
        )
    ),
    "The federal funds rate remains the quarterly arithmetic mean": bool(
        np.allclose(
            macro_unlagged["effective_federal_funds_rate"].to_numpy(float),
            quarterly_components.loc[
                FULL_SOURCE_CALENDAR,
                "effective_federal_funds_rate",
            ].to_numpy(float),
            rtol=0,
            atol=1e-12,
        )
    ),
    "No unlagged macro value is missing":
        (int(macro_unlagged.isna().sum().sum()) == 0),
    "Every unlagged macro value is finite": bool(
        np.isfinite(macro_unlagged.to_numpy(float)).all()
    ),
}

notebook_08_cell_04_gate = pd.DataFrame({
    "check": list(cell_04_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_04_checks.values()],
    "passed": list(cell_04_checks.values()),
})
macro_unlagged.reset_index().to_csv(
    NB08_OUTPUT_DIR / "macro_quarterly_unlagged_full_observed_range.csv",
    index=False,
)
notebook_08_cell_04_gate.to_csv(
    NB08_OUTPUT_DIR / "cell_04_validation_gate.csv",
    index=False,
)

print("Notebook 08 Cell 4 validation gate:")
display(notebook_08_cell_04_gate)
print("\nFull unlagged quarterly macro preview:")
display(macro_unlagged.reset_index().head(10))

failed = [name for name, passed in cell_04_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 08 Cell 4 blocked. Failed checks: " + "; ".join(failed))

print("\nCell 4 completed successfully.")

In [ ]:
# Map source quarter t−1 to reference quarter t exactly once

if not notebook_08_cell_04_gate["passed"].all():
    raise AssertionError("Cell 5 blocked because Cell 4 did not pass.")


def quarter_index(values):
    periods = pd.DatetimeIndex(values).to_period("Q")
    return pd.Series(
        periods.year * 4 + periods.quarter,
        index=getattr(values, "index", None),
        dtype="int64",
    )


macro_lag_mapping_audit = macro_unlagged.reset_index().copy()
macro_lag_mapping_audit["report_date"] = (
    macro_lag_mapping_audit["source_quarter"] + pd.offsets.QuarterEnd(1)
)
macro_lag_mapping_audit["source_quarter_index"] = quarter_index(
    macro_lag_mapping_audit["source_quarter"]
).to_numpy()
macro_lag_mapping_audit["report_quarter_index"] = quarter_index(
    macro_lag_mapping_audit["report_date"]
).to_numpy()
macro_lag_mapping_audit["quarter_index_gap"] = (
    macro_lag_mapping_audit["report_quarter_index"]
    - macro_lag_mapping_audit["source_quarter_index"]
)
macro_lag_mapping_audit["macro_lag_applications"] = MACRO_LAG_APPLICATIONS
macro_lag_mapping_audit = macro_lag_mapping_audit[
    [
        "report_date",
        "source_quarter",
        "source_quarter_index",
        "report_quarter_index",
        "quarter_index_gap",
        "macro_lag_applications",
        "real_gdp_growth",
        "unemployment_rate",
        "effective_federal_funds_rate",
        "house_price_growth",
    ]
].copy()

macro_full_lagged = macro_lag_mapping_audit[
    [
        "report_date",
        "real_gdp_growth",
        "unemployment_rate",
        "effective_federal_funds_rate",
        "house_price_growth",
    ]
].rename(columns={
    "real_gdp_growth": "real_gdp_growth_lag1",
    "unemployment_rate": "unemployment_rate_lag1",
    "effective_federal_funds_rate": "effective_federal_funds_rate_lag1",
    "house_price_growth": "house_price_growth_lag1",
})
macro_full_lagged = macro_full_lagged[["report_date"] + MACRO_PREDICTORS]

macro_rows_by_date = macro_full_lagged.groupby("report_date").size()
candidate_reference_coverage_audit = pd.DataFrame({
    "report_date": CANDIDATE_REFERENCE_CALENDAR
})
candidate_reference_coverage_audit["macro_rows_for_reference_date"] = (
    candidate_reference_coverage_audit["report_date"]
    .map(macro_rows_by_date)
    .fillna(0)
    .astype(int)
)
candidate_reference_coverage_audit = candidate_reference_coverage_audit.merge(
    macro_lag_mapping_audit[
        [
            "report_date",
            "source_quarter",
            "source_quarter_index",
            "report_quarter_index",
            "quarter_index_gap",
            "macro_lag_applications",
        ]
    ],
    on="report_date",
    how="left",
    validate="one_to_one",
)
macro_candidate_lagged = pd.DataFrame({
    "report_date": CANDIDATE_REFERENCE_CALENDAR
}).merge(
    macro_full_lagged,
    on="report_date",
    how="left",
    validate="one_to_one",
)

cell_05_checks = {
    "The full lagged panel exactly matches the dynamically derived full calendar":
        pd.DatetimeIndex(macro_full_lagged["report_date"]).equals(FULL_REPORT_CALENDAR),
    "The candidate handoff exactly matches the restricted Notebook 07 calendar":
        pd.DatetimeIndex(macro_candidate_lagged["report_date"]).equals(
            CANDIDATE_REFERENCE_CALENDAR
        ),
    "Every candidate reference date has exactly one macro row": bool(
        candidate_reference_coverage_audit["macro_rows_for_reference_date"].eq(1).all()
    ),
    "Every candidate macro row is measured at source quarter t−1": bool(
        candidate_reference_coverage_audit["quarter_index_gap"].eq(1).all()
    ),
    "The macro lag was applied exactly once": bool(
        candidate_reference_coverage_audit["macro_lag_applications"].eq(1).all()
    ),
    "No second macro lag is introduced for the lagged-bank V2 design":
        (MACRO_LAG_APPLICATIONS == 1),
    "The candidate handoff contains one date and four predictors":
        (list(macro_candidate_lagged.columns) == ["report_date"] + MACRO_PREDICTORS),
    "No candidate macro value is missing":
        (int(macro_candidate_lagged[MACRO_PREDICTORS].isna().sum().sum()) == 0),
    "Every candidate macro value is finite": bool(
        np.isfinite(macro_candidate_lagged[MACRO_PREDICTORS].to_numpy(float)).all()
    ),
    "No full-panel report quarter is duplicated":
        (int(macro_full_lagged["report_date"].duplicated().sum()) == 0),
    "No candidate report quarter is duplicated":
        (int(macro_candidate_lagged["report_date"].duplicated().sum()) == 0),
}

notebook_08_cell_05_gate = pd.DataFrame({
    "check": list(cell_05_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_05_checks.values()],
    "passed": list(cell_05_checks.values()),
})

FULL_MACRO_PARQUET_PATH = (
    NB08_OUTPUT_DIR / "macro_predictors_lagged_full_observed_range.parquet"
)
FULL_MACRO_CSV_PATH = (
    NB08_OUTPUT_DIR / "macro_predictors_lagged_full_observed_range.csv"
)
CANDIDATE_MACRO_PARQUET_PATH = (
    NB08_OUTPUT_DIR / "macro_predictors_lagged_candidate_reference.parquet"
)
CANDIDATE_MACRO_CSV_PATH = (
    NB08_OUTPUT_DIR / "macro_predictors_lagged_candidate_reference.csv"
)
MACRO_LAG_AUDIT_PATH = (
    NB08_OUTPUT_DIR / "macro_predictors_lag_mapping_audit.csv"
)
CANDIDATE_COVERAGE_AUDIT_PATH = (
    NB08_OUTPUT_DIR / "cell_05_candidate_reference_coverage_audit.csv"
)

macro_full_lagged.to_parquet(
    FULL_MACRO_PARQUET_PATH,
    index=False,
    compression="snappy",
)
macro_full_lagged.to_csv(FULL_MACRO_CSV_PATH, index=False)
macro_candidate_lagged.to_parquet(
    CANDIDATE_MACRO_PARQUET_PATH,
    index=False,
    compression="snappy",
)
macro_candidate_lagged.to_csv(CANDIDATE_MACRO_CSV_PATH, index=False)
macro_lag_mapping_audit.to_csv(MACRO_LAG_AUDIT_PATH, index=False)
candidate_reference_coverage_audit.to_csv(
    CANDIDATE_COVERAGE_AUDIT_PATH,
    index=False,
)
notebook_08_cell_05_gate.to_csv(
    NB08_OUTPUT_DIR / "cell_05_validation_gate.csv",
    index=False,
)

print("Notebook 08 Cell 5 validation gate:")
display(notebook_08_cell_05_gate)
print("\nCandidate lagged macroeconomic handoff preview:")
display(macro_candidate_lagged.head(10))
print("\nCandidate source-quarter and quarter-index preview:")
display(candidate_reference_coverage_audit.head(10))

failed = [name for name, passed in cell_05_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 08 Cell 5 blocked. Failed checks: " + "; ".join(failed))

print("\nCell 5 completed successfully.")
print("Notebook 09 primary macro handoff:", CANDIDATE_MACRO_PARQUET_PATH)
print("Full observed-range macro panel:", FULL_MACRO_PARQUET_PATH)
print("Source-quarter mapping audit:", MACRO_LAG_AUDIT_PATH)

In [ ]:
# Audit extremes, revised vintages and nominal publication timing

if not notebook_08_cell_05_gate["passed"].all():
    raise AssertionError("Cell 6 blocked because Cell 5 did not pass.")

cell_06_start_hash = int(
    pd.util.hash_pandas_object(macro_candidate_lagged, index=False).sum()
)

distribution_records = []
extreme_records = []
for predictor in MACRO_PREDICTORS:
    values = macro_candidate_lagged[predictor].astype(float)
    q1 = float(values.quantile(0.25))
    q3 = float(values.quantile(0.75))
    iqr = q3 - q1
    lower_fence = q1 - 3 * iqr
    upper_fence = q3 + 3 * iqr
    minimum_index = values.idxmin()
    maximum_index = values.idxmax()

    distribution_records.append({
        "predictor": predictor,
        "observations": len(values),
        "mean": float(values.mean()),
        "standard_deviation": float(values.std()),
        "minimum": float(values.min()),
        "minimum_quarter": macro_candidate_lagged.loc[minimum_index, "report_date"],
        "p25": q1,
        "median": float(values.median()),
        "p75": q3,
        "maximum": float(values.max()),
        "maximum_quarter": macro_candidate_lagged.loc[maximum_index, "report_date"],
        "three_iqr_lower_fence": lower_fence,
        "three_iqr_upper_fence": upper_fence,
    })

    extreme_mask = values.lt(lower_fence) | values.gt(upper_fence)
    for row_index in macro_candidate_lagged.index[extreme_mask]:
        report_date = macro_candidate_lagged.loc[row_index, "report_date"]
        source_quarter = candidate_reference_coverage_audit.loc[
            candidate_reference_coverage_audit["report_date"].eq(report_date),
            "source_quarter",
        ].iloc[0]
        extreme_records.append({
            "report_date": report_date,
            "source_quarter": source_quarter,
            "predictor": predictor,
            "value": float(macro_candidate_lagged.loc[row_index, predictor]),
            "lower_fence": lower_fence,
            "upper_fence": upper_fence,
            "treatment": "Retained unchanged; historical extreme documented for audit",
        })

cell_06_distribution_summary = pd.DataFrame(distribution_records)
cell_06_extreme_observations = pd.DataFrame(
    extreme_records,
    columns=[
        "report_date",
        "source_quarter",
        "predictor",
        "value",
        "lower_fence",
        "upper_fence",
        "treatment",
    ],
)

publication_records = []
publication_detail_records = []
for row in macro_definition.itertuples(index=False):
    nominal_release_dates = (
        candidate_reference_coverage_audit["source_quarter"]
        + pd.to_timedelta(row.nominal_publication_lag_days, unit="D")
    )
    availability_margin_days = (
        candidate_reference_coverage_audit["report_date"] - nominal_release_dates
    ).dt.days
    normally_available = availability_margin_days.ge(0)
    publication_records.append({
        "series_id": row.series_id,
        "predictor": row.final_predictor,
        "source_frequency": row.raw_frequency,
        "nominal_publication_lag_days": row.nominal_publication_lag_days,
        "nominal_publication_basis": row.nominal_publication_basis,
        "minimum_days_available_before_reference_quarter_end": int(
            availability_margin_days.min()
        ),
        "t_minus_1_normally_published_before_quarter_end_t": bool(
            normally_available.all()
        ),
        "timing_qualification": (
            "Nominal timing check only; current revised vintages are used and exact "
            "historical release timestamps are not reconstructed."
        ),
        "official_timing_source": row.official_timing_source,
        "paired_downstream_audit": (
            "Report beside the Notebook 09 bank filing-availability audit"
        ),
    })
    for audit_row, nominal_date, margin, available in zip(
        candidate_reference_coverage_audit.itertuples(index=False),
        nominal_release_dates,
        availability_margin_days,
        normally_available,
    ):
        publication_detail_records.append({
            "series_id": row.series_id,
            "predictor": row.final_predictor,
            "report_date": audit_row.report_date,
            "source_quarter": audit_row.source_quarter,
            "nominal_publication_date": nominal_date,
            "nominal_days_available_before_report_date": int(margin),
            "normally_available_before_report_date": bool(available),
        })

cell_06_nominal_publication_timing_audit = pd.DataFrame(publication_records)
cell_06_nominal_publication_timing_detail = pd.DataFrame(publication_detail_records)

cell_06_vintage_note = pd.DataFrame({
    "topic": [
        "Data vintage",
        "Revision treatment",
        "Interpretive limitation",
        "Macro timing protection",
        "Bank-lag interaction",
        "Nominal publication timing",
        "Extreme observations",
        "Winsorisation",
        "Missing-value treatment",
    ],
    "documented_treatment": [
        "The four local FRED CSV files are current-vintage snapshots identified by filename and SHA-256 hash.",
        "The series may contain revisions made after historical reference dates; real-time vintages are not used.",
        "Results use subsequently revised macroeconomic history and are not a fully real-time forecasting exercise.",
        "Each macro value is mapped once from source quarter t−1 to reference quarter t.",
        "Bank predictors are shifted separately to t−1 downstream; the macro panel must not receive another lag.",
        "Publication availability is checked using nominal release lags, not exact historical release timestamps.",
        "Three-IQR extremes are listed for audit and retained unchanged.",
        "No value is winsorised in Notebook 08; fold-specific preprocessing remains downstream.",
        "No value is imputed in Notebook 08; complete candidate-calendar coverage is required.",
    ],
})

cell_06_end_hash = int(
    pd.util.hash_pandas_object(macro_candidate_lagged, index=False).sum()
)
all_extremes_traceable = True
for record in extreme_records:
    match = macro_candidate_lagged.loc[
        macro_candidate_lagged["report_date"].eq(record["report_date"]),
        record["predictor"],
    ]
    if len(match) != 1 or not np.isclose(
        float(match.iloc[0]),
        record["value"],
        rtol=0,
        atol=1e-12,
    ):
        all_extremes_traceable = False
        break

cell_06_checks = {
    "All four predictors have a complete distribution audit":
        (set(cell_06_distribution_summary["predictor"]) == set(MACRO_PREDICTORS)),
    "Every flagged extreme traces exactly to the candidate handoff":
        bool(all_extremes_traceable),
    "The extreme-value audit did not modify the macro handoff":
        (cell_06_start_hash == cell_06_end_hash),
    "Current-vintage revised data are explicitly documented": bool(
        cell_06_vintage_note["documented_treatment"].str.contains(
            "revis",
            case=False,
        ).any()
    ),
    "Real-time vintages are explicitly not claimed": bool(
        cell_06_vintage_note["documented_treatment"].str.contains(
            "not a fully real-time",
            case=False,
            regex=False,
        ).any()
    ),
    "All four nominal publication-lag checks were produced":
        (len(cell_06_nominal_publication_timing_audit) == 4),
    "Every t−1 macro value would normally be published before quarter-end t": bool(
        cell_06_nominal_publication_timing_audit[
            "t_minus_1_normally_published_before_quarter_end_t"
        ].all()
    ),
    "Publication timing is qualified as nominal rather than exact historical timing": bool(
        cell_06_nominal_publication_timing_audit["timing_qualification"]
        .str.contains("Nominal timing check only", regex=False)
        .all()
    ),
    "The macro timing audit is paired with the downstream bank filing audit": bool(
        cell_06_nominal_publication_timing_audit["paired_downstream_audit"]
        .str.contains("bank filing-availability audit", regex=False)
        .all()
    ),
    "No winsorisation or imputation was performed": bool(
        cell_06_vintage_note["documented_treatment"]
        .str.contains("No value is winsorised", regex=False).any()
        and cell_06_vintage_note["documented_treatment"]
        .str.contains("No value is imputed", regex=False).any()
    ),
}

notebook_08_cell_06_gate = pd.DataFrame({
    "check": list(cell_06_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_06_checks.values()],
    "passed": list(cell_06_checks.values()),
})
cell_06_distribution_summary.to_csv(
    NB08_OUTPUT_DIR / "cell_06_macro_distribution_and_extrema.csv",
    index=False,
)
cell_06_extreme_observations.to_csv(
    NB08_OUTPUT_DIR / "cell_06_three_iqr_extreme_observations.csv",
    index=False,
)
cell_06_nominal_publication_timing_audit.to_csv(
    NB08_OUTPUT_DIR / "cell_06_nominal_macro_publication_timing_audit.csv",
    index=False,
)
cell_06_nominal_publication_timing_detail.to_csv(
    NB08_OUTPUT_DIR / "cell_06_nominal_macro_publication_timing_detail.csv",
    index=False,
)
cell_06_vintage_note.to_csv(
    NB08_OUTPUT_DIR / "cell_06_data_vintage_and_treatment_note.csv",
    index=False,
)
notebook_08_cell_06_gate.to_csv(
    NB08_OUTPUT_DIR / "cell_06_validation_gate.csv",
    index=False,
)

print("Notebook 08 Cell 6 validation gate:")
display(notebook_08_cell_06_gate)
print("\nDistribution and extreme-quarter summary:")
display(cell_06_distribution_summary)
print("\nThree-IQR observations retained for audit:")
display(cell_06_extreme_observations)
print("\nNominal macro publication-timing audit:")
with pd.option_context("display.max_colwidth", None):
    display(cell_06_nominal_publication_timing_audit)
print("\nData-vintage and treatment note:")
with pd.option_context("display.max_colwidth", None):
    display(cell_06_vintage_note)

failed = [name for name, passed in cell_06_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 08 Cell 6 blocked. Failed checks: " + "; ".join(failed))

print("\nCell 6 completed successfully. No macro value was removed or modified.")

In [ ]:
# Final validation gate and definitive Notebook 09 handoff

if not notebook_08_cell_06_gate["passed"].all():
    raise AssertionError("Cell 7 blocked because Cell 6 did not pass.")

full_parquet_check = pd.read_parquet(FULL_MACRO_PARQUET_PATH)
candidate_parquet_check = pd.read_parquet(CANDIDATE_MACRO_PARQUET_PATH)
candidate_csv_check = pd.read_csv(
    CANDIDATE_MACRO_CSV_PATH,
    parse_dates=["report_date"],
)

full_parquet_exact = True
candidate_parquet_exact = True
candidate_csv_exact = True
try:
    pd.testing.assert_frame_equal(
        full_parquet_check,
        macro_full_lagged,
        check_dtype=True,
        check_exact=True,
    )
except AssertionError:
    full_parquet_exact = False
try:
    pd.testing.assert_frame_equal(
        candidate_parquet_check,
        macro_candidate_lagged,
        check_dtype=True,
        check_exact=True,
    )
except AssertionError:
    candidate_parquet_exact = False
try:
    pd.testing.assert_frame_equal(
        candidate_csv_check,
        macro_candidate_lagged,
        check_dtype=False,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12,
    )
except AssertionError:
    candidate_csv_exact = False

all_previous_gates = pd.concat([
    notebook_08_cell_01_gate,
    notebook_08_cell_02_gate,
    notebook_08_cell_03_gate,
    notebook_08_cell_04_gate,
    notebook_08_cell_05_gate,
    notebook_08_cell_06_gate,
], ignore_index=True)

cell_07_checks = {
    "The full macro panel matches its dynamically derived calendar":
        (len(macro_full_lagged) == len(FULL_REPORT_CALENDAR)),
    "The candidate handoff matches the restricted Notebook 07 calendar":
        (len(macro_candidate_lagged) == len(CANDIDATE_REFERENCE_CALENDAR)),
    "Exactly one Notebook 07 reference quarter was excluded":
        (len(EXCLUDED_REFERENCE_CALENDAR) == 1),
    "The excluded Notebook 07 reference quarter is 2001Q1": bool(
        EXCLUDED_REFERENCE_CALENDAR.equals(
            pd.DatetimeIndex([pd.Timestamp("2001-03-31")])
        )
    ),
    "The candidate handoff begins at 2001Q2":
        (macro_candidate_lagged["report_date"].min() == REFERENCE_CALENDAR_START),
    "The candidate handoff ends at the last eligible Notebook 07 reference date":
        (
            macro_candidate_lagged["report_date"].max()
            == CANDIDATE_REFERENCE_CALENDAR.max()
        ),
    "Every candidate date has exactly one macro row": bool(
        candidate_reference_coverage_audit["macro_rows_for_reference_date"].eq(1).all()
    ),
    "Every candidate macro source quarter is exactly t−1": bool(
        candidate_reference_coverage_audit["quarter_index_gap"].eq(1).all()
    ),
    "The macro lag was applied once and only once":
        (MACRO_LAG_APPLICATIONS == 1),
    "The candidate handoff contains one date and four lagged predictors":
        (list(macro_candidate_lagged.columns) == ["report_date"] + MACRO_PREDICTORS),
    "No candidate macro value is missing":
        (int(macro_candidate_lagged[MACRO_PREDICTORS].isna().sum().sum()) == 0),
    "Every candidate macro value is finite": bool(
        np.isfinite(macro_candidate_lagged[MACRO_PREDICTORS].to_numpy(float)).all()
    ),
    "The primary candidate Parquet output exists":
        CANDIDATE_MACRO_PARQUET_PATH.exists(),
    "The full observed-range Parquet output exists":
        FULL_MACRO_PARQUET_PATH.exists(),
    "The source-quarter mapping audit exists": MACRO_LAG_AUDIT_PATH.exists(),
    "The candidate coverage audit exists": CANDIDATE_COVERAGE_AUDIT_PATH.exists(),
    "The nominal publication-timing audit exists": (
        NB08_OUTPUT_DIR / "cell_06_nominal_macro_publication_timing_audit.csv"
    ).exists(),
    "The full Parquet output re-read exactly": full_parquet_exact,
    "The candidate Parquet output re-read exactly": candidate_parquet_exact,
    "The candidate CSV output re-read with the same values": candidate_csv_exact,
    "All previous Notebook 08 gates passed": bool(all_previous_gates["passed"].all()),
}

notebook_08_cell_07_gate = pd.DataFrame({
    "check": list(cell_07_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_07_checks.values()],
    "passed": list(cell_07_checks.values()),
})

notebook_08_final_summary = pd.DataFrame({
    "measure": [
        "Analysis version",
        "Raw FRED series",
        "Reference calendar start",
        "Latest common complete-quarter candidate",
        "First incomplete source quarter",
        "Common complete macro source-data end",
        "Full panel truncated at first internal gap",
        "Derived full macro report end",
        "Full macro report quarters",
        "Notebook 07 reference dates",
        "Notebook 07 dates excluded before V2 start",
        "Excluded reference quarters",
        "Candidate reference quarters",
        "First candidate report date",
        "Last candidate report date",
        "Candidate handoff columns",
        "Lagged macro predictors",
        "Missing candidate macro values",
        "Duplicate candidate report dates",
        "Macro lag applications",
        "Timing rule",
        "GDP transformation",
        "Unemployment transformation",
        "Federal funds transformation",
        "House-price transformation",
        "Data vintage",
        "Publication timing qualification",
        "Nominal publication checks passing",
        "Bank filing-audit pairing",
        "Extreme-value treatment",
        "Notebook 09 primary macro handoff",
        "Full observed-range macro panel",
        "Source-quarter mapping audit",
    ],
    "value": [
        ANALYSIS_VERSION,
        4,
        REFERENCE_CALENDAR_START,
        LATEST_COMMON_COMPLETE_CANDIDATE,
        FIRST_INCOMPLETE_SOURCE_QUARTER,
        COMMON_SOURCE_DATA_END,
        bool(pd.notna(FIRST_INCOMPLETE_SOURCE_QUARTER)),
        FULL_REPORT_END,
        len(macro_full_lagged),
        len(NOTEBOOK_07_REFERENCE_CALENDAR),
        len(EXCLUDED_REFERENCE_CALENDAR),
        ",".join(str(date.to_period("Q")) for date in EXCLUDED_REFERENCE_CALENDAR),
        len(macro_candidate_lagged),
        macro_candidate_lagged["report_date"].min(),
        macro_candidate_lagged["report_date"].max(),
        macro_candidate_lagged.shape[1],
        len(MACRO_PREDICTORS),
        int(macro_candidate_lagged[MACRO_PREDICTORS].isna().sum().sum()),
        int(macro_candidate_lagged["report_date"].duplicated().sum()),
        MACRO_LAG_APPLICATIONS,
        "All four predictors use source quarter t−1 for reference quarter t exactly once",
        "Quarter-on-quarter percentage growth; not annualised",
        "Quarterly arithmetic mean",
        "Quarterly arithmetic mean",
        "Quarter-on-quarter percentage growth; not annualised",
        "Current-vintage FRED snapshots; subsequently revised rather than real-time vintages",
        "Nominal only; exact historical release timestamps are not reconstructed",
        int(cell_06_nominal_publication_timing_audit[
            "t_minus_1_normally_published_before_quarter_end_t"
        ].sum()),
        "Macro publication audit is to be reported beside Notebook 09 bank filing availability",
        "Three-IQR extremes documented and retained unchanged",
        CANDIDATE_MACRO_PARQUET_PATH.name,
        FULL_MACRO_PARQUET_PATH.name,
        MACRO_LAG_AUDIT_PATH.name,
    ],
})

notebook_08_output_manifest = pd.DataFrame({
    "file": [
        CANDIDATE_MACRO_PARQUET_PATH.name,
        CANDIDATE_MACRO_CSV_PATH.name,
        FULL_MACRO_PARQUET_PATH.name,
        FULL_MACRO_CSV_PATH.name,
        MACRO_LAG_AUDIT_PATH.name,
        CANDIDATE_COVERAGE_AUDIT_PATH.name,
        "cell_06_nominal_macro_publication_timing_audit.csv",
        "cell_06_nominal_macro_publication_timing_detail.csv",
        "cell_06_data_vintage_and_treatment_note.csv",
        "cell_03_incomplete_source_quarter_audit.csv",
        "cell_07_final_validation_gate.csv",
        "notebook_08_final_handoff_summary.csv",
    ],
    "role": [
        "Primary Notebook 09 macro handoff",
        "Readable primary handoff",
        "Full observed-range macro panel",
        "Readable full observed-range panel",
        "Full source-quarter and quarter-index audit",
        "Notebook 07 candidate-calendar coverage audit",
        "Nominal macro publication-timing summary",
        "Nominal publication timing by candidate quarter",
        "Revision and treatment limitation",
        "Incomplete-quarter and continuous-range audit",
        "Final validation gate",
        "Final handoff summary",
    ],
    "leakage_sensitivity": [
        "Predictors only; source dates excluded",
        "Predictors only; source dates excluded",
        "Predictors only; source dates excluded",
        "Predictors only; source dates excluded",
        "Audit only; contains source quarters",
        "Audit only; contains source quarters",
        "Audit only",
        "Audit only; contains nominal release dates",
        "Control documentation",
        "Audit only",
        "Control",
        "Control",
    ],
})

notebook_08_cell_07_gate.to_csv(
    NB08_OUTPUT_DIR / "cell_07_final_validation_gate.csv",
    index=False,
)
notebook_08_final_summary.to_csv(
    NB08_OUTPUT_DIR / "notebook_08_final_handoff_summary.csv",
    index=False,
)
notebook_08_output_manifest.to_csv(
    NB08_OUTPUT_DIR / "notebook_08_output_manifest.csv",
    index=False,
)

print("Notebook 08 Cell 7 validation gate:")
display(notebook_08_cell_07_gate)
print("\nNotebook 08 final handoff summary:")
with pd.option_context("display.max_colwidth", None):
    display(notebook_08_final_summary)
print("\nNotebook 08 output manifest:")
with pd.option_context("display.max_colwidth", None):
    display(notebook_08_output_manifest)
print("\nPrimary candidate macro handoff preview:")
display(macro_candidate_lagged.head(10))

failed = [name for name, passed in cell_07_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 08 final gate failed: " + "; ".join(failed))

print("\nCell 7 completed successfully.")
print("Notebook 08 is PASS / LOCKED.")
print("\nNotebook 09 primary macro handoff:")
print(CANDIDATE_MACRO_PARQUET_PATH)
print("\nFull observed-range macro panel:")
print(FULL_MACRO_PARQUET_PATH)
print("\nNotebook 09 can now verify bank and macro t−1 timing side by side.")